# Jupyter-Scatter Landscape

In [ ]:
import celldega as dega
import ipywidgets
import jscatter
import numpy as np
import pandas as pd
from ipywidgets import HBox, Layout, jslink
import scanpy as sc
import ipywidgets as ipw

In [ ]:
adata_cat = sc.read_h5ad('data/visium-hd_data/Visium_HD_Human_Colon_Cancer/crc_cluster.h5ad')
adata = sc.read_10x_h5('data/visium-hd_data/Visium_HD_Human_Colon_Cancer/raw_feature_cell_matrix.h5')

adata_cat.uns['leiden_colors'] = adata_cat.uns['clusters_colors']
adata.obs['leiden'] = adata_cat.obs['leiden']

In [ ]:
base_url = 'https://raw.githubusercontent.com/SharkieJones/celldega_Visium-HD_hCRC/refs/heads/main/Visium_HD_Human_Colon_Cancer_2025-10-25_2um'

landscape = dega.viz.Landscape(
    technology='Visium-HD',
    base_url = base_url,
    height=500, 
    max_tiles_to_view=5, 
    adata=adata_cat,
)

landscape.layout = Layout(
    width="500px", 
)

In [ ]:
import ipywidgets as w
import pandas as pd
import jscatter

# --- Build DataFrame ---------------------------------------------------------

df = pd.DataFrame({
    "x": adata[:, "B2M"].X.toarray().ravel(),
    "y": adata[:, "IGKC"].X.toarray().ravel(),
}, index=adata.obs_names)

df["color"] = adata.obs["leiden"].astype("category")

# --- Build Scatter -----------------------------------------------------------

scatter = jscatter.Scatter(
    data=df,
    x="x",
    y="y",
    color="color",
    width=500,
    height=500,
    size=5
)

scatter.color(by="color")
scatter.data(use_index=True)

scatter_widget = scatter.show(buttons=["pan_zoom", "lasso", "reset"])

# --- Selection sync: scatter → landscape -------------------------------------

def selection_change_handler(change):
    landscape.selected_cells = scatter.selection().values.tolist()

scatter.widget.observe(selection_change_handler, names=["selection"])

# --- Build Search Boxes (Combobox with autocomplete) ------------------------

# Make sure these are strings and sorted for nicer search
gene_list = sorted([str(g) for g in adata.var_names])

x_box = w.Combobox(
    options=gene_list,
    value="B2M",              # must be in options
    description="x:",
    ensure_option=True,       # only allow valid genes
    continuous_update=False,  # update on commit, not every keystroke
    layout=w.Layout(width="220px")
)

y_box = w.Combobox(
    options=gene_list,
    value="IGKC",
    description="y:",
    ensure_option=True,
    continuous_update=False,
    layout=w.Layout(width="220px")
)

def update_x(change):
    gene = change["new"]
    if gene in adata.var_names:   # small safety check
        s = pd.Series(adata[:, gene].X.toarray().ravel(), index=df.index)
        scatter.x(s)

def update_y(change):
    gene = change["new"]
    if gene in adata.var_names:
        s = pd.Series(adata[:, gene].X.toarray().ravel(), index=df.index)
        scatter.y(s)

x_box.observe(update_x, names="value")
y_box.observe(update_y, names="value")

controls = w.HBox([x_box, y_box])

# --- Compose Layout -----------------------------------------------------------

# Left column = Landscape
landscape.layout = w.Layout(width="650px", height="650px")

# Right column = controls above scatter
left_panel = w.VBox(
    [controls, scatter_widget],
    layout=w.Layout(width="650px", height="650px")
)

ui = w.HBox(
    [landscape, left_panel],
    layout=w.Layout(height="650px")
)

ui
